## Apply all Cell-Health Models to Training and Testing Sets

**Gregory Way, 2019**

In [1]:
import os
import pandas as pd
from joblib import load

from scripts.ml_utils import load_train_test, load_models

In [2]:
%matplotlib inline

In [3]:
def apply_model(model, feature, train_x, test_x):
    """
    Apply model to training and testing matrix
    """
    pred_train_df = (
        pd.DataFrame(model.predict(train_x), columns=["score"])
        .assign(profiles=train_x.index,
                Metadata_data_type="train",
                model=feature)
    )
    pred_test_df = (
        pd.DataFrame(model.predict(test_x), columns=["score"])
        .assign(profiles=test_x.index,
                Metadata_data_type="test",
                model=feature)
    )

    pred_df = pd.concat([pred_train_df, pred_test_df]).reset_index(drop=True)
    return pred_df

def sample_squared_error(scores, y):
    """
    Calculate the squared error per sample depending on model scores
    """
    metadata_cols = [x for x in scores.columns if x.startswith("Metadata_")]
    scores_values = scores.drop(metadata_cols, axis="columns")
    
    all_squared_error = {}
    for cell_health_feature in scores_values.columns:
        y_subset_df = y.loc[:, cell_health_feature].dropna().T
        scores_subset = scores_values.loc[:, cell_health_feature].reindex(y_subset_df.index).T

        squared_error = (y_subset_df - scores_subset) ** 2
        all_squared_error[cell_health_feature] = squared_error
    
    return pd.DataFrame(all_squared_error).reindex(scores.index)

## 1) Load Models and Model Coefficients

For real data and shuffled model data.

In [4]:
consensus = "modz"

In [5]:
model_dict, model_coef = load_models(consensus=consensus)
shuffle_model_dict, shuffle_model_coef = load_models(shuffle=True, consensus=consensus)

In [6]:
# Load Metadata Mapping File
data_dir = os.path.join("..", "1.generate-profiles", "data")
file = os.path.join(data_dir, "profile_id_metadata_mapping.tsv")
metadata_df = pd.read_csv(file, sep='\t')

metadata_df.head()

,Metadata_profile_id,Metadata_cell_line,Metadata_pert_name
0,profile_0,A549,AKT1-1
1,profile_1,A549,AKT1-2
2,profile_2,A549,ARID1B-1
3,profile_3,A549,ARID1B-2
4,profile_4,A549,ATF4-1


## 2) Load Training and Testing Data

In [7]:
x_train_df, x_test_df, y_train_df, y_test_df = load_train_test(drop_metadata=True, consensus=consensus)

## 3) Output Model Coefficients

In [8]:
# Extract all model coefficients and output to file
coef_df = pd.DataFrame(model_coef)
coef_df.index = x_test_df.columns
coef_df.index.name = "features"

file = os.path.join("results",
                    "all_model_coefficients_{}.tsv".format(consensus))
coef_df.to_csv(file, sep='\t', index=True)

print(coef_df.shape)
coef_df.head(2)

(410, 70)


,cell_health_modz_target_cc_late_mitosis_n_spots_h2ax_per_nucleus_area_mean,cell_health_modz_target_cc_all_nucleus_roundness_mean,cell_health_modz_target_cc_polyploid_n_objects,cell_health_modz_target_cc_g1_n_objects,cell_health_modz_target_cc_mitosis_n_objects,cell_health_modz_target_vb_num_live_cells,cell_health_modz_target_vb_percent_live,cell_health_modz_target_vb_live_cell_roundness,cell_health_modz_target_cc_all_large_notround_polynuclear_mean,cell_health_modz_target_cc_all_high_h2ax,...,cell_health_modz_target_cc_s_intensity_nucleus_area_mean,cell_health_modz_target_cc_g2_high_h2ax,cell_health_modz_target_cc_late_mitosis_n_spots_h2ax_mean,cell_health_modz_target_cc_cc_high_h2ax,cell_health_modz_target_vb_percent_dead_only,cell_health_modz_target_cc_s_high_h2ax,cell_health_modz_target_cc_cc_n_spots_h2ax_per_nucleus_area_mean,cell_health_modz_target_cc_s_n_spots_h2ax_per_nucleus_area_mean,cell_health_modz_target_cc_g1_plus_g2_count,cell_health_modz_target_vb_live_cell_width_length
features,,,,,,,,,,,,,,,,,,,,,
level_0,-0.00075,-0.000281,-0.00164,0.001576,-0.000921,0.001855,0.000654,0.001797,-0.002833,-0.001236,...,0.000218,-0.001302,-0.000983,-0.001451,-0.000416,-0.001193,-0.001727,-0.001199,-0.000875,0.001693
Image_Count_Cells,-0.00000,0.000000,-0.00000,0.027126,0.007871,0.039636,0.000000,0.031014,0.000000,-0.011351,...,0.041936,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.005328,0.058293


In [9]:
# Extract all model coefficients and output to file
shuffle_coef_df = pd.DataFrame(shuffle_model_coef)
shuffle_coef_df.index = x_test_df.columns
shuffle_coef_df.index.name = "features"

file = os.path.join("results",
                    "all_model_coefficients_shuffled_{}.tsv".format(consensus))
shuffle_coef_df.to_csv(file, sep='\t', index=True)

print(shuffle_coef_df.shape)
shuffle_coef_df.head(2)

(410, 70)


,cell_health_modz_target_cc_all_n_objects,cell_health_modz_target_cc_g1_high_h2ax,cell_health_modz_target_vb_num_live_cells,cell_health_modz_target_cc_all_high_h2ax,cell_health_modz_target_cc_g1_n_objects,cell_health_modz_target_cc_g1_n_spots_h2ax_per_nucleus_area_mean,cell_health_modz_target_cc_late_mitosis_n_spots_h2ax_mean,cell_health_modz_target_cc_cc_n_spots_h2ax_per_nucleus_area_mean,cell_health_modz_target_cc_cc_g1,cell_health_modz_target_cc_polynuclear_n_spots_h2ax_per_nucleus_area_mean,...,cell_health_modz_target_cc_all_n_spots_h2ax_per_nucleus_area_mean,cell_health_modz_target_cc_s_intensity_nucleus_area_sum,cell_health_modz_target_cc_early_mitosis_n_spots_h2ax_mean,cell_health_modz_target_cc_g1_n_spots_h2ax_mean,cell_health_modz_target_cc_cc_mitosis,cell_health_modz_target_cc_polynuclear_n_objects,cell_health_modz_target_cc_polynuclear_high_h2ax,cell_health_modz_target_cc_s_intensity_nucleus_area_mean,cell_health_modz_target_cc_early_mitosis_n_objects,cell_health_modz_target_cc_early_mitosis_high_h2ax
features,,,,,,,,,,,,,,,,,,,,,
level_0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,...,-0.0,-0.000000,0.0,0.0,0.000000,-0.0,-0.00031,0.0,-0.0,-0.0
Image_Count_Cells,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,...,-0.0,0.001885,0.0,-0.0,-0.000213,0.0,-0.00000,0.0,-0.0,0.0


## 4) Apply all models

For real and shuffled data.

In [10]:
all_scores = []
all_shuffle_scores = []
for cell_health_feature in model_dict.keys():
    # Apply Real Model Classifiers
    model_clf = model_dict[cell_health_feature]
    pred_df = apply_model(model=model_clf,
                          feature=cell_health_feature,
                          train_x=x_train_df,
                          test_x=x_test_df)
    all_scores.append(pred_df)
    
    # Apply Shuffled Model Classifiers
    shuffle_model_clf = shuffle_model_dict[cell_health_feature]
    shuffle_pred_df = apply_model(model=shuffle_model_clf,
                                  feature=cell_health_feature,
                                  train_x=x_train_df,
                                  test_x=x_test_df)
    all_shuffle_scores.append(shuffle_pred_df)

## 5) Concatenate scores with Metadata

In [12]:
# Concatenate real data scores
all_scores = (
    pd.concat(all_scores)
    .reset_index(drop=True)
    .pivot_table(index=["profiles", "Metadata_data_type"],
                 columns="model",
                 values="score")
    .reset_index()
)

# Convert profiles column to string type to match Metadata_profile_id
all_scores['profiles'] = all_scores['profiles'].astype(str)

# Now merge
all_scores = (
    metadata_df.merge(
        all_scores,
        left_on="Metadata_profile_id",
        right_on="profiles"
    )
    .drop("profiles", axis="columns")
)

In [13]:

all_scores.index = all_scores.Metadata_profile_id
all_scores = all_scores.drop("Metadata_profile_id", axis="columns")

# Remove prefix of variable columns
strip_text = "cell_health_{}_target_".format(consensus)
all_scores.columns = [x.replace(strip_text, "") for x in all_scores.columns]

# Output file
file = os.path.join(
    "results", "all_model_predictions_{}.tsv".format(consensus)
)
all_scores.to_csv(file, sep='\t', index=True)

print(all_scores.shape)
all_scores.head(2)

(0, 73)


,Metadata_cell_line,Metadata_pert_name,Metadata_data_type,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,cc_all_n_spots_h2ax_per_nucleus_area_mean,cc_all_nucleus_area_mean,...,vb_num_live_cells,vb_percent_all_apoptosis,vb_percent_caspase_dead_only,vb_percent_dead,vb_percent_dead_only,vb_percent_early_apoptosis,vb_percent_late_apoptosis,vb_percent_live,vb_ros_back_mean,vb_ros_mean
Metadata_profile_id,,,,,,,,,,,,,,,,,,,,,


In [14]:
# Concatenate shuffled data scores
all_shuffle_scores = (
    pd.concat(all_shuffle_scores)
    .reset_index(drop=True)
    .pivot_table(index=["profiles", "Metadata_data_type"],
                 columns="model",
                 values="score")
    .reset_index()
)

# Convert profiles column to string type to match Metadata_profile_id
all_shuffle_scores['profiles'] = all_shuffle_scores['profiles'].astype(str)

all_shuffle_scores = (
    metadata_df.merge(all_shuffle_scores,
                      left_on="Metadata_profile_id",
                      right_on="profiles")
    .drop("profiles", axis="columns")
)

# Remove prefix of variable columns
strip_text = "cell_health_{}_target_".format(consensus)
all_shuffle_scores.columns = [x.replace(strip_text, "") for x in all_shuffle_scores.columns]

# Output file
file = os.path.join(
    "results", "all_model_predictions_shuffled_{}.tsv".format(consensus)
)
all_shuffle_scores.to_csv(file, sep='\t', index=True)

print(all_shuffle_scores.shape)
all_shuffle_scores.head(2)

(0, 74)


,Metadata_profile_id,Metadata_cell_line,Metadata_pert_name,Metadata_data_type,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,cc_all_n_spots_h2ax_per_nucleus_area_mean,...,vb_num_live_cells,vb_percent_all_apoptosis,vb_percent_caspase_dead_only,vb_percent_dead,vb_percent_dead_only,vb_percent_early_apoptosis,vb_percent_late_apoptosis,vb_percent_live,vb_ros_back_mean,vb_ros_mean


## 6) Calculate the Squared Error of Individual Samples

For real and shuffled data

In [15]:
y_df = pd.concat([y_train_df, y_test_df]).reindex(all_scores.index)

print(y_df.shape)
y_df.head(2)

(0, 70)


,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,cc_all_n_spots_h2ax_per_nucleus_area_mean,cc_all_nucleus_area_mean,cc_all_nucleus_roundness_mean,cc_cc_early_mitosis,cc_cc_g1,...,vb_num_live_cells,vb_percent_all_apoptosis,vb_percent_caspase_dead_only,vb_percent_dead,vb_percent_dead_only,vb_percent_early_apoptosis,vb_percent_late_apoptosis,vb_percent_live,vb_ros_back_mean,vb_ros_mean
Metadata_profile_id,,,,,,,,,,,,,,,,,,,,,


In [16]:
all_score_error = sample_squared_error(scores=all_scores, y=y_df)

all_score_error = (
    metadata_df.merge(
        all_score_error,
        left_on="Metadata_profile_id",
        right_index=True
    )
)

# Output file
file = os.path.join(
    "results", "all_model_sample_squared_error_{}.tsv".format(consensus)
)
all_score_error.to_csv(file, sep='\t', index=False)

print(all_score_error.shape)
all_score_error.head(2)

(0, 73)


,Metadata_profile_id,Metadata_cell_line,Metadata_pert_name,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,cc_all_n_spots_h2ax_per_nucleus_area_mean,cc_all_nucleus_area_mean,...,vb_num_live_cells,vb_percent_all_apoptosis,vb_percent_caspase_dead_only,vb_percent_dead,vb_percent_dead_only,vb_percent_early_apoptosis,vb_percent_late_apoptosis,vb_percent_live,vb_ros_back_mean,vb_ros_mean


In [17]:



all_shuffle_score_error = sample_squared_error(scores=all_shuffle_scores, y=y_df)

all_shuffle_score_error = (
    metadata_df.merge(
        all_shuffle_score_error,
        left_on="Metadata_profile_id",
        right_index=True
    )
)

# Output file
file = os.path.join(
    "results", "all_model_sample_squared_error_shuffled_{}.tsv".format(consensus)
)
all_shuffle_score_error.to_csv(file, sep='\t', index=False)

print(all_shuffle_score_error.shape)
all_shuffle_score_error.head()

(0, 73)


,Metadata_profile_id,Metadata_cell_line,Metadata_pert_name,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,cc_all_n_spots_h2ax_per_nucleus_area_mean,cc_all_nucleus_area_mean,...,vb_num_live_cells,vb_percent_all_apoptosis,vb_percent_caspase_dead_only,vb_percent_dead,vb_percent_dead_only,vb_percent_early_apoptosis,vb_percent_late_apoptosis,vb_percent_live,vb_ros_back_mean,vb_ros_mean
